# MEOK: strict public-edge replay
Version3 adds actual calls from Kaggle to the repaired public Cloudflare research preview. It reuses the original980-parameter model unchanged, checks all560rows and6,720raw/constrained/rule values, rejects incomplete results and records failure controls.

No new training, GPU, private credentials, private data or GSPC score. The preview expires12November2026 00:00UTC; after expiry the expected response is a refusal, not a service guarantee. Versions1–2 and their outcomes remain historical.


In [ ]:
from pathlib import Path
import urllib.request, hashlib, json, importlib.util, sys
ROOT=Path('/kaggle/working/meok-cross-runtime-v3'); ROOT.mkdir(exist_ok=True)
PIN='125fb63c9fe65df88a7558203b7ae4b282a02a32'
LOCKS={'replay.py': 'a1f8f74fe3f7a52db111c874b740668998ae672d7a4a159f41aeec2daf287be3', 'response_contract.py': '66280a99580113bda59531e78a980496253a4897a16021e1182fdd7369cd11e7', 'source-lock.json': '442d0563cad67a60eb9e78611ff9ead144d10610c2e165afcb0822d051eb70cc', 'test_response_contract.py': '9a04b32b3a85d8e7ea8a000363640dbdba452a8cbfdc62c5a9ca7a28dfe53bc2'}
class NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self,*args,**kwargs):return None
opener=urllib.request.build_opener(NoRedirect)
def fetch_source(path,digest):
    url='https://raw.githubusercontent.com/CSOAI-ORG/councilof-ai/'+PIN+'/harness/meok-cross-runtime/'+path
    with opener.open(url,timeout=25) as response:raw=response.read(300001)
    assert len(raw)<=300000 and hashlib.sha256(raw).hexdigest()==digest
    target=ROOT/path;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
for name,digest in LOCKS.items():fetch_source(name,digest)
source_lock=json.loads((ROOT/'source-lock.json').read_text())
for name,item in source_lock['files'].items():
    assert Path(name).name==name
    fetch_source('vendor/'+name,item['sha256'])
(ROOT/'completion').mkdir(exist_ok=True)
sys.path.insert(0,str(ROOT))
import test_response_contract
contract=test_response_contract.run()
assert contract['negative_cases']==40 and contract['retained_rows_verified']==560
import replay
EDGE='https://0e29bc3e-csoai-process-replay.nicholastempleman.workers.dev'
report=replay.run('Kaggle-v3-public-Cloudflare-strict-acceptance',ROOT/'results',EDGE)
assert report['state']=='CROSS_RUNTIME_PARITY_PASSED'
assert report['edge']['rows_compared']==560 and report['edge']['strict_response_completion_checked']
assert report['edge']['including_exact_rule_values_verified']==6720 and report['edge']['requests']==78
(ROOT/'SOURCE.json').write_text(json.dumps({'source_commit':PIN,'files_sha256':LOCKS,'public_edge':EDGE,'model_training':False,'uses_private_credentials':False,'scope':'Dated public research replay before the fixed12November2026 00:00UTC expiry; not a permanent service.'},indent=2)+'\n')
print('KAGGLE_V3_STRICT_PUBLIC_EDGE_COMPLETED')


## Evidence
Download `meok-cross-runtime-v3/results/report.json`, `request-ledger.json`, `predictions.jsonl`, and `completion/response-contract-tests.json`. This establishes numerical/contract portability on an already-public synthetic bank, not improved accuracy or independent organisational evaluation. The model retains its approximately0.75worst component error.
